# 04 · Regime Analysis
Transition matrices, regime durations, AR vs tau conditioning, country ranking.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from data.synthetic import generate_em_universe
from modules.turbulence import compute_turbulence_index
from modules.absorption import compute_absorption_ratio

uni    = generate_em_universe(seed=42)
panel  = uni.panel
fx_ret = uni.fx
eq_ret = uni.equity

WINDOW = 252; SLOW_W = 504; LAM = 0.94
REGIMES = ['Calm','Elevated','Turbulent','Crisis']

turb = compute_turbulence_index(panel,  window=WINDOW, min_periods=60,
                                vol_standardize=True, slow_window=SLOW_W)
ar   = compute_absorption_ratio(fx_ret, window=WINDOW, min_periods=60, lam=LAM)

fx_cols = list(fx_ret.columns)
eq_cols = list(eq_ret.columns)
country_turb = {}
for i, col in enumerate(fx_cols):
    ct = pd.concat([fx_ret[[col]], eq_ret[[eq_cols[i]]]], axis=1)
    country_turb[col] = compute_turbulence_index(
        ct, window=WINDOW, min_periods=60, vol_standardize=True, slow_window=SLOW_W)

print(f'Regime: {turb.current_regime()}   AR: {float(ar.absorption_ratio.dropna().iloc[-1]):.4f}')


## 2 · Regime Transition Matrix  P(regime_{t+1} | regime_t)


In [ ]:
reg = turb.regime.dropna()

trans = pd.DataFrame(0, index=REGIMES, columns=REGIMES, dtype=float)
for t in range(len(reg) - 1):
    r_t, r_t1 = reg.iloc[t], reg.iloc[t+1]
    if r_t in REGIMES and r_t1 in REGIMES:
        trans.loc[r_t, r_t1] += 1
trans_pct = trans.div(trans.sum(axis=1), axis=0).round(3)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(trans_pct, annot=True, fmt='.3f', cmap='Blues',
            vmin=0, vmax=1, linewidths=0.4, ax=ax, annot_kws={'size': 11})
ax.set_title('Empirical Regime Transition Matrix P(t+1 | t)')
ax.set_xlabel('Regime at t+1'); ax.set_ylabel('Regime at t')
plt.tight_layout(); plt.show()
print(trans_pct.to_string())


## 3 · Average Regime Duration


In [ ]:
durations = {r: [] for r in REGIMES}
cur_r, cur_n = None, 0
for r in reg:
    if r == cur_r: cur_n += 1
    else:
        if cur_r in REGIMES and cur_n > 0: durations[cur_r].append(cur_n)
        cur_r, cur_n = r, 1
if cur_r in REGIMES and cur_n > 0: durations[cur_r].append(cur_n)

rows = []
for r in REGIMES:
    vals = durations[r]
    if vals:
        rows.append({'Regime': r, 'N episodes': len(vals),
                     'Mean (d)': round(np.mean(vals),1),
                     'Median (d)': round(np.median(vals),1),
                     'Max (d)': max(vals)})
print(pd.DataFrame(rows).set_index('Regime').to_string())


## 4 · AR Level Conditioned on Turbulence Regime

High AR + high tau: systemic (co-movement).  High tau + low AR: idiosyncratic.


In [ ]:
common = turb.regime.index.intersection(ar.absorption_ratio.index)
df_j   = pd.DataFrame({'regime': turb.regime.loc[common],
                        'AR': ar.absorption_ratio.loc[common]}).dropna()

reg_colors = {'Calm':'#2ecc71','Elevated':'#f39c12','Turbulent':'#e67e22','Crisis':'#e74c3c'}
fig, ax = plt.subplots(figsize=(8, 4))
for i, r in enumerate(REGIMES):
    vals = df_j.loc[df_j['regime'] == r, 'AR']
    if len(vals) == 0: continue
    bp = ax.boxplot(vals, positions=[i], widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=reg_colors[r], alpha=0.7),
               medianprops=dict(color='white', lw=2),
               whiskerprops=dict(color=reg_colors[r]),
               capprops=dict(color=reg_colors[r]),
               flierprops=dict(marker='.', color=reg_colors[r], alpha=0.3))
ax.set_xticks(range(len(REGIMES))); ax.set_xticklabels(REGIMES)
ax.set_ylabel('Absorption Ratio')
ax.set_title('AR Distribution by Turbulence Regime')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()
print(df_j.groupby('regime')['AR'].agg(['mean','median','std']).round(4).to_string())


## 5 · Country Stress Rank — Monthly Heatmap


In [ ]:
monthly = pd.DataFrame({
    c: country_turb[c].turbulence.resample('MS').mean() for c in fx_cols})

monthly_z = (monthly - monthly.mean(axis=1).values[:,None]) / (
    monthly.std(axis=1).values[:,None] + 1e-8)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(monthly_z.T.values, aspect='auto', cmap='RdYlGn_r',
               vmin=-2, vmax=2, interpolation='nearest')
ax.set_yticks(range(len(fx_cols))); ax.set_yticklabels(fx_cols)
yr_ticks = [i for i, d in enumerate(monthly.index) if d.month == 1]
ax.set_xticks(yr_ticks)
ax.set_xticklabels([monthly.index[i].year for i in yr_ticks])
plt.colorbar(im, ax=ax, label='Turbulence z-score (cross-country)')
ax.set_title('Country Stress Rank — Monthly Turbulence Z-score')
plt.tight_layout(); plt.show()


## 6 · Country tau Correlation with Panel tau


In [ ]:
ROLL = 63
panel_tau = turb.turbulence.dropna()
acolors = ['#00d4aa','#e74c3c','#f39c12','#9b59b6','#3498db']

fig, ax = plt.subplots(figsize=(13, 4))
for i, col in enumerate(fx_cols):
    ct_tau = country_turb[col].turbulence.dropna()
    common = panel_tau.index.intersection(ct_tau.index)
    rc = panel_tau.loc[common].rolling(ROLL).corr(ct_tau.loc[common])
    ax.plot(rc.index, rc, label=col, color=acolors[i], lw=1.1, alpha=0.85)
ax.axhline(0,   color='white',  lw=0.5, alpha=0.3)
ax.axhline(0.7, color='orange', ls='--', lw=0.7, alpha=0.5)
ax.set_ylabel(f'{ROLL}d rolling corr with panel tau')
ax.set_title('Country Turbulence Correlation with Panel tau')
ax.legend(fontsize=9); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()
